# Text Data Analysis: 2D Parameter Matrix

This notebook analyzes parameter combinations to find which features affect image scores.
Creates matrices showing how different parameter pairs correlate with scores.

In [ ]:
import sys
import os
from pathlib import Path

# Setup path
root_path = str(Path(os.getcwd()).parents[2])
sys.path.insert(0, root_path)

print(f"Root path: {root_path}")
print(f"Python path configured")


Root path: e:\ComfyUI\custom_nodes\comfyui-image-scorer
Python path configured


In [ ]:
from shared.paths import (
    text_data_file,
    scores_file,
    vectors_file,
    matrix_analysis_file,
    matrix_summary_file,
    training_dir,
)
from shared.io import load_single_jsonl
from shared.training.matrix_analysis import MatrixAnalyzer
from pathlib import Path
import json

# Ensure training directory exists
Path(training_dir).mkdir(parents=True, exist_ok=True)

# Load data
print("Loading data...")
text_data = load_single_jsonl(text_data_file)
scores = load_single_jsonl(scores_file)

print(f"Text data records: {len(text_data)}")
print(f"Score records: {len(scores)}")

if len(text_data) != len(scores):
    min_len = min(len(text_data), len(scores))
    print(f"WARNING: Data length mismatch! Using {min_len} records")
    text_data = text_data[:min_len]
    scores = scores[:min_len]

# Display sample
if text_data:
    print(f"\nSample text record keys: {list(text_data[0].keys())}")
    print(f"Sample score: {scores[0]}")


Loading data...
Text data records: 24451
Score records: 24451

Sample text record keys: ['lora', 'sampler', 'scheduler', 'model', 'steps', 'clip_skip', 'original_width', 'original_height', 'final_width', 'final_height', 'cfg', 'original_aspect_ratio', 'final_aspect_ratio', 'sharpness', 'noise_score', 'contrast', 'colorfulness', 'artifact_score', 'edge_density', 'texture_lbp', 'lora_weight', 'negative_prompt', 'positive_prompt']
Sample score: 1.0


In [ ]:
# Initialize analyzer
# memory_limit controls max unique parameters to track
MEMORY_LIMIT = 15000  # Adjust based on RAM available

print("Initializing Matrix Analyzer...")
analyzer = MatrixAnalyzer(scores, text_data, memory_limit=MEMORY_LIMIT)

print(f"Matrix Analyzer initialized")
print(f"Memory limit: {MEMORY_LIMIT} unique parameters")
print(f"Ready to build comprehensive parameter matrix")


Initializing Matrix Analyzer...
Matrix Analyzer initialized
Memory limit: 100000000 unique parameters
Ready to build comprehensive parameter matrix


## 2D Matrix Analysis

Analyze how combinations of parameters affect scores. Each cell shows score statistics for that parameter combination.

In [ ]:
# Build comprehensive matrix combining ALL parameters
print("\n" + "=" * 80)
print("BUILDING COMPREHENSIVE 2D PARAMETER MATRIX")
print("=" * 80)
print("This creates a symmetric matrix of all parameter values from text data")
print("Each cell contains scores for records that have both parameters\n")

try:
    analyzer.build_matrix()
    print(f"✓ Matrix building complete")
except Exception as e:
    print(f"Error during matrix building: {e}")
    import traceback

    traceback.print_exc()



BUILDING COMPREHENSIVE 2D PARAMETER MATRIX
This creates a symmetric matrix of all parameter values from text data
Each cell contains scores for records that have both parameters

Processing 24451 text records...


Extracting parameters: 100%|██████████| 24451/24451 [00:02<00:00, 11626.42 records/s]


Found 12855 unique parameters
Building parameter co-occurrence matrix...


Building matrix: 100%|██████████| 24451/24451 [00:24<00:00, 992.08 records/s] 

Matrix built: 12855x12855 parameters
✓ Matrix building complete


In [ ]:
# Calculate statistics for all matrix cells
print("\n" + "=" * 80)
print("CALCULATING STATISTICS FOR MATRIX CELLS")
print("=" * 80 + "\n")

try:
    stats: dict[tuple[int, int], dict[str, float]] = analyzer.calculate_statistics(
        min_count=1000
    )
    print(f"✓ Statistics calculated for {len(stats)} cells")
except Exception as e:
    print(f"Error during statistics calculation: {e}")
    import traceback

    traceback.print_exc()



CALCULATING STATISTICS FOR MATRIX CELLS



Flattening Matrix: 100%|██████████| 82631940/82631940 [01:05<00:00, 1262153.00cells/s]


Stats: Kept 59912 cells, Dropped 82572028 cells (Min Count: 100)
Calculating Statistics via Polars (Multithreaded)...


Building Final Dict: 100%|██████████| 59912/59912 [00:00<00:00, 337730.16it/s]


✓ Statistics calculated for 118770 cells


In [21]:
# Display matrix summary
print("\n" + "=" * 80)
print("MATRIX SUMMARY")
print("=" * 80)

try:
    summary = analyzer.get_matrix_summary()
    for key, value in summary.items():
        print(f"  {key}: {value}")

    size = analyzer.get_matrix_size()
    print(f"\n  Matrix Size: {size[0]}x{size[1]} parameters")
except Exception as e:
    print(f"Error getting summary: {e}")
    import traceback

    traceback.print_exc()



MATRIX SUMMARY
  total_parameters: 12855
  matrix_cells: 118770
  total_score_entries: 71469481.0
  mean_of_means: 2.89482475230928
  loaded_records: 24451
  loaded_vectors: 24451

  Matrix Size: 12855x12855 parameters


## Export Results

Save all matrices to JSON for further analysis

In [22]:
# Export matrix to JSON
print("\n" + "=" * 80)
print("EXPORTING MATRIX ANALYSIS")
print("=" * 80 + "\n")

try:
    analyzer.export_to_json(matrix_analysis_file)
    print(f"✓ Matrix exported to: {matrix_analysis_file}")
    print(f"✓ Total cells with data: {len(analyzer.cell_stats)}")

    # Also export summary
    summary = analyzer.get_matrix_summary()
    with open(matrix_summary_file, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"✓ Summary exported to: {matrix_summary_file}")

except Exception as e:
    print(f"Error during export: {e}")
    import traceback

    traceback.print_exc()



EXPORTING MATRIX ANALYSIS



Exporting to JSON: 100%|██████████| 118770/118770 [00:00<00:00, 1846710.28 cells/s]


✓ Exported 0 cell statistics to e:\ComfyUI\custom_nodes\comfyui-image-scorer\output\training\matrix_analysis_results.jsonl
✓ Matrix exported to: e:\ComfyUI\custom_nodes\comfyui-image-scorer\output\training\matrix_analysis_results.jsonl
✓ Total cells with data: 118770
✓ Summary exported to: e:\ComfyUI\custom_nodes\comfyui-image-scorer\output\training\matrix_analysis_summary.json
